In [1]:
#import libraries
import pandas as pd
import requests
import os
import re
from sklearn.preprocessing import LabelEncoder

In [2]:
#define project path
project_path = "/Users/karenzhu/Desktop/Projects/MAIN_structure_project/work/SP001_050/"
ECOD_cluster_path = os.path.join(project_path, "SP020/ECOD/ECOD_easy_cluster_cluster.tsv")
ECOD_annotations_path = os.path.join(project_path, "SP017/ECOD_files/ECOD_ManualRep_domain_annotations.csv")

ECOD_clusters_df = pd.read_csv(ECOD_cluster_path , sep='\t', header=None, names = ['Representative', 'Member'])
ECOD_annotations_df = pd.read_csv(ECOD_annotations_path, dtype = str)


# 1. Create a text file of uid's that only have a single chain

In [3]:
#remove entries with multiple chains and output to a text file
ECOD_annotations_no_chains_df = ECOD_annotations_df[ECOD_annotations_df["chain"]!="."]
ECOD_annotations_no_chains_path = os.path.join(project_path, "SP017/ECOD_ManualRep_domain_annotations_single_chains.txt")
ECOD_annotations_no_chains_df[['uid']].to_csv(ECOD_annotations_no_chains_path, header=False, index=False)

# 2. Analyze foldseek clustering results - check for uid's with multiple chains

In [4]:
#read in the results from foldseek easy clustering
ECOD_clusters_df = pd.read_csv(ECOD_cluster_path , sep='\t', header=None, names = ['Representative', 'Member'])

#read in ECOD annotations and select relevant columns
ECOD_annotations_df = ECOD_annotations_df[['uid', 'arch_name', 'x_name', 'h_name', 't_name', 'f_name']]

In [6]:
#determine how many domains are separated into multiple files of different chains. 
#There are 21195 pdb files before separating into chains

#find the number of pdb files after foldseek easy clustering
ECOD_unique_pdbs = set(list(ECOD_clusters_df['Member'].unique()))

#find the number of pdb files separated into different chains
ECOD_unique_pdbs = list(ECOD_unique_pdbs)

pdb_multiple_chains = []
for pdb_name in ECOD_unique_pdbs:
    match_list = re.findall("[0-9]*.pdbnum_[a-zA-Z]", pdb_name)
    if len(match_list)!=0:
        for match in match_list:
            pdb_multiple_chains.append(match)

#find the number of pdbs with different chains
pdb_multiple_chains_no_base = []
regex = re.compile(r'^([0-9]+)\.pdbnum_[a-zA-Z]$')

for pdb_name in pdb_multiple_chains:
    match = regex.match(pdb_name)
    if match:
        pdb_id = match.group(1)
        pdb_multiple_chains_no_base.append(pdb_id)
unique_pdb_with_chains = set(pdb_multiple_chains_no_base)

print("Number of pdbs after separating domains into different chains:  ", len(ECOD_unique_pdbs))
print("Number of pdb files with chains after foldseek easy cluster: ", len(pdb_multiple_chains))
print("Number of pdbs with different chains: ", len(unique_pdb_with_chains ))

Number of pdbs after separating domains into different chains:   21032
Number of pdb files with chains after foldseek easy cluster:  116
Number of pdbs with different chains:  115


In [7]:
#find the pdb file that is separated into different chains
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    
    for item in lst:
        if item in seen:
            duplicates.add(item)
        else:
            seen.add(item)
    
    return list(duplicates)

print("Find the domain pdb file separted into chains: ",find_duplicates(pdb_multiple_chains_no_base))

Find the domain pdb file separted into chains:  ['002387813']


ECOD includes have domains that have multiple chains (~200). I removed the domains with multiple chains from the annotation file.
I reran foldseek easy cluster, however 002387813 is separated into chains A and C. For example, 002387813.pdbnum_A & 002387813.pdbnum_C
Looking at the annotation file, it only mentions chain C. 

Action: I will keep both files for now.

# 3. Create annotation file for the cluster rep/member

In [8]:
#for the rep/cluster name, split into the uid and the chain. This will help us match to the annotations
pdb_name_pattern = r'(\d+)\.pdbnum_?([a-zA-Z]?)'

ECOD_clusters_df[['Rep_uid', 'Rep_chain']] = ECOD_clusters_df['Representative'].str.extract(pdb_name_pattern)

ECOD_clusters_df[['Member_uid', 'Member_chain']] = ECOD_clusters_df['Member'].str.extract(pdb_name_pattern)

In [143]:
#join the annotations to the clusters
ECOD_clusters_df = ECOD_clusters_df.merge(ECOD_annotations_df, left_on = 'Rep_uid',
                                         right_on = 'uid', how = 'left')

In [144]:
columns_to_prefix = ['arch_name', 'x_name', 'h_name', 't_name', 'f_name' ]
ECOD_clusters_df = ECOD_clusters_df.drop(columns = ['uid'])
ECOD_clusters_df = ECOD_clusters_df.rename(columns={col: 'Rep_' + col for col in columns_to_prefix}) 

In [145]:
ECOD_clusters_df = ECOD_clusters_df.merge(ECOD_annotations_df, left_on = 'Member_uid',
                                         right_on = 'uid', how = 'left')

ECOD_clusters_df = ECOD_clusters_df.drop(columns = ['uid'])
ECOD_clusters_df = ECOD_clusters_df.rename(columns={col: 'Member_' + col for col in columns_to_prefix}) 

In [146]:
#label encode the text classification and assign psuedo numbers to create a psuedo number classification
unique_classification_vals = pd.unique(
    ECOD_clusters_df[['Rep_arch_name', 'Rep_x_name', 'Rep_h_name',
       'Rep_t_name', 'Rep_f_name', 'Member_arch_name', 'Member_x_name',
       'Member_h_name', 'Member_t_name', 'Member_f_name']].values.ravel()
    
)

label_encoder = LabelEncoder()
label_encoder.fit(unique_classification_vals)

#label ecnode columns
cols_to_encode = ['Rep_arch_name', 'Rep_x_name', 'Rep_h_name',
       'Rep_t_name', 'Rep_f_name', 'Member_arch_name', 'Member_x_name',
       'Member_h_name', 'Member_t_name', 'Member_f_name']

for column in cols_to_encode:
   ECOD_clusters_df[column] = label_encoder.transform(ECOD_clusters_df[column])

In [147]:
#after label encoding, make psuedo code's for reps
ECOD_clusters_df['Rep_psuedo_code_fam'] = (
    ECOD_clusters_df['Rep_arch_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_x_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_h_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_t_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_f_name'].astype(str)
)

ECOD_clusters_df['Rep_psuedo_code_top'] = (
    ECOD_clusters_df['Rep_arch_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_x_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_h_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_t_name'].astype(str) 
)

ECOD_clusters_df['Rep_psuedo_code_homol'] = (
    ECOD_clusters_df['Rep_arch_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_x_name'].astype(str) + "." +
    ECOD_clusters_df['Rep_h_name'].astype(str) 
)

#after label encoding, make psuedo code's for members
ECOD_clusters_df['Member_psuedo_code_fam'] = (
    ECOD_clusters_df['Member_arch_name'].astype(str) + "." +
    ECOD_clusters_df['Member_x_name'].astype(str) + "." +
    ECOD_clusters_df['Member_h_name'].astype(str) + "." +
    ECOD_clusters_df['Member_t_name'].astype(str) + "." +
    ECOD_clusters_df['Member_f_name'].astype(str)
)

ECOD_clusters_df['Member_psuedo_code_top'] = (
    ECOD_clusters_df['Member_arch_name'].astype(str) + "." +
    ECOD_clusters_df['Member_x_name'].astype(str) + "." +
    ECOD_clusters_df['Member_h_name'].astype(str) + "." +
    ECOD_clusters_df['Member_t_name'].astype(str)
)

ECOD_clusters_df['Member_psuedo_code_homol'] = (
    ECOD_clusters_df['Member_arch_name'].astype(str) + "." +
    ECOD_clusters_df['Member_x_name'].astype(str) + "." +
    ECOD_clusters_df['Member_h_name'].astype(str) 
)

In [148]:
#see if the two cath code match for representative and member. assign 1 if match and 0 if mismatch
ECOD_clusters_df['Code_match_status_fam'] = ECOD_clusters_df['Rep_psuedo_code_fam'] == ECOD_clusters_df['Member_psuedo_code_fam']
ECOD_clusters_df['Code_match_status_fam'] = ECOD_clusters_df['Code_match_status_fam'].astype(int)

ECOD_clusters_df['Code_match_status_top'] = ECOD_clusters_df['Rep_psuedo_code_top'] == ECOD_clusters_df['Member_psuedo_code_top']
ECOD_clusters_df['Code_match_status_top'] = ECOD_clusters_df['Code_match_status_top'].astype(int)

ECOD_clusters_df['Code_match_status_homol'] = ECOD_clusters_df['Rep_psuedo_code_homol'] == ECOD_clusters_df['Member_psuedo_code_homol']
ECOD_clusters_df['Code_match_status_homol'] = ECOD_clusters_df['Code_match_status_homol'].astype(int)

In [149]:
print("Number of mismatched classifications: ",ECOD_clusters_df[ECOD_clusters_df['Code_match_status_fam']==0].shape[0])
print("Number of mismatched classifications: ",ECOD_clusters_df[ECOD_clusters_df['Code_match_status_top']==0].shape[0])
print("Number of mismatched classifications: ",ECOD_clusters_df[ECOD_clusters_df['Code_match_status_homol']==0].shape[0])

Number of mismatched classifications:  5468
Number of mismatched classifications:  427
Number of mismatched classifications:  104


In [150]:
#decode the label encoded dataframe

ECOD_clusters_encoded_df = ECOD_clusters_df.copy()

# Reverse encode specified columns
for column in cols_to_encode:
    ECOD_clusters_df[column] = label_encoder.inverse_transform(ECOD_clusters_df[column])


ECOD_clusters_decoded_df = ECOD_clusters_df.copy()

In [151]:
#for each representative/cluster, determine the fraction of matched members to its rep
ECOD_calc_matches_df = ECOD_clusters_decoded_df.groupby('Representative').agg(
    Fam_cluster_matches_num=('Code_match_status_fam', 'sum'),
    Cluster_size=('Code_match_status_fam', 'size')
).reset_index()

ECOD_calc_matches_df['Fam_match_frac'] = ECOD_calc_matches_df['Fam_cluster_matches_num']/ECOD_calc_matches_df['Cluster_size']
ECOD_calc_matches_df['Fam_match_frac'] = ECOD_calc_matches_df['Fam_match_frac'].round(decimals=2)

ECOD_calc_matches_df.head()

,Representative,Fam_cluster_matches_num,Cluster_size,Fam_match_frac
0,000000005.pdbnum,1,1,1.00
1,000000028.pdbnum,1,1,1.00
2,000000032.pdbnum,4,23,0.17
3,000000034.pdbnum,1,1,1.00
4,000000035.pdbnum,1,1,1.00


In [152]:
ECOD_clusters_decoded_df = pd.merge(ECOD_clusters_decoded_df, ECOD_calc_matches_df, how='left', on='Representative')


In [153]:
#output the annotations file
annotation_file_path = os.path.join(project_path, "SP020/ECOD/ECOD_foldseek_cluster_annotations.csv")
ECOD_clusters_decoded_df.to_csv(annotation_file_path, index=False)

# 4. Create a text file of cluster representatives. This is to create a nonredundant dataset for ECOD.

In [156]:
ECOD_reps_only_df = ECOD_clusters_df[ECOD_clusters_df['Representative'] == ECOD_clusters_df['Member']]
ECOD_reps_only_df = ECOD_reps_only_df[['Rep_uid']]

cluster_rep_file_name = os.path.join(project_path, 'SP020/ECOD/ECOD_foldseek_cluster_reps.txt')
ECOD_reps_only_df.to_csv(cluster_rep_file_name, header=False, index = False)